# 8장 실습 ③ — MNIST와 표준 데이터셋

**PyTorch 판**

도형 대신 진짜 손글씨 숫자로 같은 것을 해 봅니다.

> ⚠ 내려받기가 막힌 환경에서는 도형 데이터로 자동 대체됩니다.
> Kaggle이나 Colab에서 돌리시면 진짜 MNIST가 받아집니다.

## 8.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 8.1 도형 데이터

28×28 회색조 영상에 원·사각형·삼각형 중 하나가 그려져 있습니다.
**위치가 매번 다릅니다.** 그것이 이 장의 핵심입니다.

In [ ]:
# 도형 데이터 — 인터넷 없이 만든다. 그리고 **위치를 마음대로 흔들 수 있다.**
# MNIST는 숫자가 대체로 가운데 있어서 CNN이 왜 필요한지가 잘 안 드러난다.
x, y = data.shapes(n=6000, seed=42, shift=6)
s = data.split(x, y, val_ratio=0.15, test_ratio=0.15, seed=42)
print(s.summary())

fig = plot.image_grid(s.x_train, s.y_train, n=24, cols=8,
                      class_names=list(data.SHAPE_CLASSES))
plt.show()

## 8.2 학습 함수 — 여기만 판마다 다릅니다

`kind` 로 DNN / CNN / 선형 모델을 만듭니다.
**PyTorch 판에서 `permute` 한 줄이 더 있는 것**에 주목하십시오 — 채널 순서가
다르기 때문입니다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

dlbook.set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

def to_nchw(a):
    """(N, H, W, C) → (N, C, H, W). PyTorch는 채널이 앞이다."""
    return torch.tensor(np.asarray(a), dtype=torch.float32).permute(0, 3, 1, 2)

def train(kind, split=None, units=(256, 128), conv_act="relu", pool="max",
          head_act="relu", n_classes=3, epochs=15, bs=64, lr=0.001, seed=42):
    """모델을 만들어 학습시키고 (시험 정확도, 파라미터 수)를 돌려준다.

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    sp = split if split is not None else s
    dlbook.set_seed(seed)
    H, W, C = sp.x_train.shape[1:]
    act = {"relu": nn.ReLU, None: None}

    if kind == "dnn":
        ls, prev = [nn.Flatten()], H * W * C
        for u in units:
            ls += [nn.Linear(prev, u), nn.ReLU()]
            prev = u
    elif kind == "linear":
        ls, prev = [nn.Flatten()], H * W * C
    else:
        Pool = nn.MaxPool2d if pool == "max" else nn.AvgPool2d
        ls, ch, side = [], C, H
        for f in (16, 32):
            ls.append(nn.Conv2d(ch, f, 3, padding=1))
            if conv_act:
                ls.append(nn.ReLU())
            ls.append(Pool(2))
            ch, side = f, side // 2
        ls.append(nn.Flatten())
        prev = ch * side * side
        ls.append(nn.Linear(prev, 64))
        if head_act:
            ls.append(nn.ReLU())
        prev = 64
    ls.append(nn.Linear(prev, n_classes))
    model = nn.Sequential(*ls).to(device)

    criterion = nn.CrossEntropyLoss()          # 소프트맥스가 안에 들어 있다
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    dl = DataLoader(TensorDataset(to_nchw(sp.x_train),
                                  torch.tensor(sp.y_train, dtype=torch.long)),
                    batch_size=bs, shuffle=True)
    for _ in range(dlbook.smoke.epochs(epochs)):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(to_nchw(sp.x_test).to(device))
    pred = logits.argmax(1).cpu().numpy()
    n_params = sum(p.numel() for p in model.parameters())
    return metrics.accuracy(sp.y_test, pred), n_params

## 8.3 표준 데이터셋 불러오기

In [ ]:
# 표준 데이터셋. 내려받기가 막힌 환경(사내망·폐쇄망)에서는 도형으로 대체한다.
try:
    sm = data.mnist()
    names = [str(i) for i in range(10)]
    title = "MNIST 손글씨 숫자"
except Exception as e:
    print(f"⚠ MNIST를 받지 못했습니다 ({type(e).__name__}). 도형 데이터로 대체합니다.")
    print("  Kaggle이나 Colab에서 돌리시면 진짜 MNIST가 받아집니다.")
    xs, ys = data.shapes(6000, seed=42, shift=6)
    sm = data.split(xs, ys, val_ratio=0.15, test_ratio=0.15, seed=42)
    names = list(data.SHAPE_CLASSES)
    title = "도형 (MNIST 대체)"

print(title)
print(sm.summary())
fig = plot.image_grid(sm.x_train, sm.y_train, n=16, cols=8, class_names=names)
plt.show()

## 8.4 DNN과 CNN

In [ ]:
n_classes = int(sm.y_train.max()) + 1
for kind, name in (("dnn", "DNN"), ("cnn", "CNN")):
    acc, n_params = train(kind, split=sm, n_classes=n_classes, epochs=8)
    print(f"{name:<6}{n_params:>12,} 파라미터   시험 정확도 {acc:.4f}")
    dlbook.record(f"ch08_mnist_{kind}_acc", acc)

print()
print("→ MNIST는 숫자가 대체로 가운데 있어서 DNN도 잘합니다.")
print("→ 그래서 이 장의 논증(§8.3)은 위치를 흔들 수 있는 도형 데이터로 했습니다.")

## 정리

- **MNIST는 숫자가 대체로 가운데 있습니다.** 그래서 DNN도 잘합니다.
  §8.3의 차이가 잘 안 드러납니다.
- **CIFAR-10은 훨씬 어렵습니다.** 배경이 있고, 각도가 다르고, 색이 다양합니다.
  여기서부터는 층을 더 쌓고(배치 정규화와 함께), 증강을 켜고,
  그래도 부족하면 **전이학습**으로 갑니다. → **9장**

### 연습

1. `data.fashion_mnist()` 와 `data.cifar10()` 으로 같은 것을 해 보십시오.
   어느 것이 가장 어렵습니까.
2. CIFAR-10에서 층을 더 쌓고 배치 정규화를 넣으면 얼마나 오릅니까.
3. 틀린 예측을 `plot.image_grid(..., preds=...)` 로 그려 보십시오.
   **어떤 것들을 틀리고 있습니까.**